# Concurrent message handling (notebook)

async-kernel provides a separate message handler for each:

- channel
- msg_type
- subshell_id

As messages are received they are queued for execution by the message handler using `queue_call` on a `Caller`. 

## Shell messaging

Both `execute_request` and `com_msg` are run in the shell's thread (normally the MainThread).
All other messages on the shell channel are run in a separate _hidden thread_ "Shell-hidden".

## Control messaging

All messages on the control channel are handled in the control thread. 

In [ ]:
import threading

import ipywidgets as ipw
from aiologic import Event

from async_kernel import utils

kernel = utils.get_kernel()

## Execute request run mode

The run mode of `execute_request` can be modified to run an `execute_request` separately as a task or thread. 

There are a few options to modify the run mode.

- Metadata
- Directly in code
- tags
- Message header (in custom messages)


<div class="admonition warning">
    <p class="admonition-title">Warning</p>
    <p>Only Jupyter lab is known to allow concurrent execution of cells.</p>
</div>


### Code for example

- **This example requires ipywidgets**
- **Ensure you are running an async-kernel**

Lets define a function that we'll reuse for the remainder of the notebook.

In [ ]:
async def demo():

    print(f"Thread name: '{threading.current_thread().name}'")
    button = ipw.Button(description="Finish")
    event = Event()
    button.on_click(lambda _: event.set())
    display(button)
    await event
    button.close()
    print(f"Finished ... thread name: '{threading.current_thread().name}'")
    return "Finished"

Lets run it normally (queue)

In [ ]:
await demo()

### Run mode: task

`task` mode instructs the kernel to execute the code in a task separate to the queue, Both `task` and `thread` execute modes can be started when the kernel is *busy executing*. There is no imposed limitation on the number of tasks (or threads) that can be run concurrently.

See also the [Caller](caller.ipynb#caller) example on how to call directly.

In [ ]:
# task
# Tip: try running this cell while the previous cell is still busy.
await demo()

### Run mode: thread

In [ ]:
# This time we'll use the tag to run the cell in a Thread
await demo()

In [ ]:
# thread
%callers # magic provided by async-kernel